In [0]:
%pip install pymongo -q

import os
import pandas as pd
from pandas import json_normalize
from pymongo import MongoClient


def get_data():
    client = MongoClient(dbutils.secrets.get(scope = "bet-master-analytics-scope", key = "MONGO_CONNECTION_STRING"))
    db = client["betmaster"]
    matches = db["matches"]
    cursor = matches.find()

    # Convert to DataFrame
    df = pd.DataFrame(list(cursor))

    # Convert dataframe rows to dicts, then normalize
    df_flat = json_normalize(df.to_dict(orient="records"), sep=".")

    # Optional cleanup
    df_flat = df_flat.drop(columns=[c for c in df_flat.columns if c.strip() == ""], errors="ignore")
    df_flat["_id"] = df_flat["_id"].astype(str)
    df_flat["team.goal.home"] = pd.to_numeric(df_flat["team.goal.home"], errors="coerce")
    df_flat["team.goal.away"] = pd.to_numeric(df_flat["team.goal.away"], errors="coerce")
    df_flat["team.goalHt.home"] = pd.to_numeric(df_flat["team.goalHt.home"], errors="coerce")
    df_flat["team.goalHt.away"] = pd.to_numeric(df_flat["team.goalHt.away"], errors="coerce")
    df_flat["team.corner.home"] = pd.to_numeric(df_flat["team.corner.home"], errors="coerce")
    df_flat["team.corner.away"] = pd.to_numeric(df_flat["team.corner.away"], errors="coerce")
    dfs = spark.createDataFrame(df_flat)
    return dfs

df = get_data()

def save_data(df):
    return df.write.format("delta").mode("overwrite").saveAsTable("bet_master_analytics.matches.match_table")

if __name__ == "__main__":
    df = get_data()
    save_data(df)